# Managing Inheritance

Tips to manage Polymorphism correctly and effectively. 

#### 1.- **Pass polymorphic types by reference or pointer** so you don't accidentally copy just the base part, which is a mistake called slicing. 

#### 2.- **Favor up-casting**, which accessing a derived object for its base class definition, because that's always safe and it costs nothing. 

#### 3.- When you really do need the derived behavior back, **use a safe down-cast**, where the compiler can tell you if you've made a mistake. 

These habits keep inheritance predictable and set the stage for comparing it with other reused techniques, which we'll also discuss. **Object slicing** is a behavior that occurs when a derived class object is copied to a base class object. Because the base class object does not have the additional members of the derived class, those extra members are sliced off and lost in the copying process. This typically happens when passing objects by value.

````C++
class Shape{
public: 
    int x, y; 
};

class Circle : public Shape{
public:
    int radius;
};

// Parameter pased by value, not by reference; makes a copy!
void printShape(Shape s){
    cout << "Shape at (" << s.x <<", " << s.y << ")" << endl;
}

int main() {
    Circle c; 
    c.x = c.y = 5; 
    c.radius = 10; 
    printShape(c); // Circle object is sliced to a Shape
}
````
    

**Polymorphism relies on pointers or references**. If you accidentally pass a derived object by value where a base object is expected, then you won't get polymorphic behavior. Instead, you'll get slicing.

**Casting** is the act of converting a pointer or reference of one type to another type. In the context of class hierarchies, you will often have a base class pointer or reference that actually refers to a derived object, and you might need to convert it back to the derived type. C++ provides several casting operators, but the primary ones used with polymorphism are **static cast** and **dynamic cast**, and the **C style cast (which is not recommended)**.

**Converting a pointer or reference from a derived class to a base class is called upcasting**. This is safe and is done implicitly in C++. For example, `Animal* A = new Dog()`. No special cast syntax is needed. 

**Downcasting is converting from a base class pointer or reference to a derived class type**. This is potentially unsafe because not every animal is a dog. For example, if an animal acutally points to a cat, casting it to dog is invalid.

````C++
class Animal{
    public:
        virtual ~Animal() = default; 
};

class Dog : public Animal{
    public:
        void bark() { std::cout << "Woff!\n"; }
}; 

class cat : public Animal {};

int main(){
    Animal* a = new Cat();                 // actually a Cat
    Dog* d = statc_cast<Dog*>(a);          // cast to a Dog, this is unsafe if a is not a Dog
    d->bark();                             // Undefined behavior
}
````

Two casts that you might use in this situation are **static** and **dynamic** cast. In polymorphic class hierarchies, we often need to determine whether a base class pointer really points to a particular derived class type.

**Dynamic cast** performs a run-time check and is the fastest way to downcast. If the cast is valid, it returns a pointer to the desired type. If the cast is not valid, it returns null pointer, or it froze an exception if used with references. It incurs a small runtime cost, but this is balanced against the safety that the runtime check offers. This cast may only be used with types that have at least one virtual memeber in ther base class. 

<img src="./IMG/Polymorphism_04.png" width="400">

````C++
Dog* d1 = dynamic_cast<Dog*>(a);                 // safe
// d1 is nullptr since 'a' is not a Dog
if(!d1){
    cout << "Casting Animal* to Dog* failed (not a Dog)." << endl; 
}
````



**Static cast** operator performs a cast without any runtime type check. It simply assumes that base pointer actually points to a derived class. If it doesn't, using the result leads to undefined behavior. Static casts should be used for downcasting only when you're absolutely sure of the objects real type. For example, within certain branches of code, even though it doesn't perform any compile time or runtime checks, this operator serves to document the intentions of the coder, so it's still preferred over a *C style cast*. 

````C++
Animal* a = new Dog();           // actually a Dog
Dog* d = static_cast<Dog*>(a);   // Safe in this case
d->bark();                       // Woff!

Animal* c = new Cat(); 
Dog* d1 = static_cast<Dog*>(c); // unsafe
// d1 is not actually a valid Dog*

if(d1){
    d1->bark(); // undefined behavior
}
````


**C style cast** Reinterpret cast operator a type to a completely unrelated type. It should never be used for casting within class hierarchies because it will not adjust the pointer to account for base class offsets or RTTI. Generally, for non polymorphic casts, like casting an integer to a pointer or between unrelated pointer types, and it's inherently unsafe if used improperly. 

````C++
#include<iostream>
#include <cstdint>

void rawMemory = malloc(sizeof(int));
int* intPtr = reinterpret_cast<int*>(rawMemory);
*intPtr = 42; 
std::cout << *intPtr << '\n'; 
free(rawMemory);
````


Sometimes, you may need to **remove const from a variable to pass it to a legacy API or to modify it under certain circumstances. The constant cast operator is the only cast that can do this**. It works only on pointers or references. Be careful, though. Modifying a constant object via **const cast** is undefined behavior. Unless the original object was not declared const. 
````C++
void update(int* p){
    *p = 99; 
}

void process(const int* data){
    update(const_cast<int*>(data));  // Only ok if original wasn't cast
}
````

Using derived base PTR in c++, will perform a static cast or reinterpet cast or constant cast as needed. It does not do  a runtime check. it's not recommended to tue *C styel cast* in C++, because they're not as explicit as the C++ cast operators. It's better to use one of the C++ cast operators to make your intention clear and get safety when needed.


# Appendix

## Run-Time Type Information
RTTI is a mechanism that allows a program to determine the actual type of an object during program execution, especially when working with inheritance and polymorphism. 

The two main RTTI features are: 
- `dynamic_cast`
- `typeid`